# Door Segmentation with Grounded-SAM

Isolate a specific door from cluttered showroom photos using GroundingDINO + SAM.

**Instance:** `ml.g4dn.xlarge` recommended. CPU works but slower.

In [ ]:
!pip install -q segment-anything supervision opencv-python-headless pillow matplotlib transformers[torch]

In [ ]:
# Download SAM weights
import os
os.makedirs("weights", exist_ok=True)
!wget -q -nc -P weights https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth

In [ ]:
import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from segment_anything import sam_model_registry, SamPredictor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

In [ ]:
# Load GroundingDINO from HuggingFace (no custom CUDA ops needed)
processor = AutoProcessor.from_pretrained("IDEA-Research/grounding-dino-tiny")
grounding_model = AutoModelForZeroShotObjectDetection.from_pretrained(
    "IDEA-Research/grounding-dino-tiny"
).to(DEVICE)

# Load SAM
sam = sam_model_registry["vit_b"](checkpoint="weights/sam_vit_b_01ec64.pth").to(DEVICE)
sam_predictor = SamPredictor(sam)
print("Models loaded.")

In [ ]:
# Upload your door images to a 'doors/' folder in this notebook's directory
IMAGE_DIR = "doors"
os.makedirs(IMAGE_DIR, exist_ok=True)

images = [f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
print(f"Found {len(images)} images: {images}")

In [ ]:
def segment_door(image_path, text_prompt="door.", box_threshold=0.3):
    """Detect and segment doors using HF GroundingDINO + SAM."""
    image = Image.open(image_path).convert("RGB")
    image_np = np.array(image)

    # GroundingDINO detection
    inputs = processor(images=image, text=text_prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = grounding_model(**inputs)

    results = processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids, box_threshold=box_threshold,
        target_sizes=[image.size[::-1]]
    )[0]

    boxes = results["boxes"].cpu().numpy()
    scores = results["scores"].cpu().numpy()

    if len(boxes) == 0:
        print("No doors detected. Try lowering box_threshold.")
        return None, None, None

    # SAM segmentation for each detected box
    sam_predictor.set_image(image_np)
    masks = []
    for box in boxes:
        mask, _, _ = sam_predictor.predict(box=box, multimask_output=False)
        masks.append(mask[0])

    return image_np, masks, scores

In [ ]:
def show_results(image, masks, scores, pick_largest=True):
    """Display segmentation results."""
    if masks is None:
        return None

    if pick_largest:
        areas = [m.sum() for m in masks]
        idx = np.argmax(areas)
        masks = [masks[idx]]
        scores = [scores[idx]]

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(image)
    axes[0].set_title("Original")
    axes[0].axis("off")

    result = image.copy()
    result[~masks[0]] = 255
    axes[1].imshow(result)
    axes[1].set_title(f"Door (conf: {scores[0]:.2f})")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()
    return masks[0]

In [ ]:
# Test on first image
if images:
    test_path = os.path.join(IMAGE_DIR, images[0])
    print(f"Processing: {images[0]}")
    image, masks, scores = segment_door(test_path)
    best_mask = show_results(image, masks, scores, pick_largest=True)

In [ ]:
# Process all images and save cutouts
os.makedirs("output", exist_ok=True)

for img_name in images:
    img_path = os.path.join(IMAGE_DIR, img_name)
    print(f"\nProcessing: {img_name}")

    image, masks, scores = segment_door(img_path)
    if masks is None:
        continue

    # Pick largest door mask
    areas = [m.sum() for m in masks]
    best_mask = masks[np.argmax(areas)]

    # Save cutout with alpha channel
    rgba = np.dstack([image, (best_mask * 255).astype(np.uint8)])
    out_name = os.path.splitext(img_name)[0] + "_cutout.png"
    Image.fromarray(rgba).save(os.path.join("output", out_name))
    print(f"  Saved: output/{out_name}")

    show_results(image, masks, scores, pick_largest=True)

## Tips

- If it picks the wrong door, try `text_prompt="grey door"` or `"front door panel"`
- Lower `box_threshold` (e.g. 0.2) if doors aren't detected
- Raise it (e.g. 0.5) if too many false positives
- The `pick_largest=True` heuristic works when your target door is the most prominent in frame

## Next Steps

Once you have clean cutouts of all 4 faces:
1. Perspective-correct each face (homography warp to rectangle)
2. Build a box-shaped GLB with front/back/side textures
3. View in AR with model-viewer